In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# 1. Set up project paths

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

INPUT_FILE = (
    PROJECT_DIR / "data" / "processed" / "clean_stocks_data.csv"
)
REPORT_DIR = PROJECT_DIR / "reports"
FIGURE_DIR = PROJECT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TRADING_DAYS_PER_YEAR = 252

In [4]:
# 2. Load and validate data

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "Clean stock data not found. Complete Step 3 first."
    )

df = pd.read_csv(INPUT_FILE, parse_dates=["Date"])

required_columns = {"Date", "Close", "Ticker"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        f"Required columns: {sorted(required_columns)}"
    )

df["Ticker"] = df["Ticker"].astype(str).str.upper().str.strip()
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

df = df.dropna(subset=["Date", "Close", "Ticker"])

if (df["Close"] <= 0).any():
    raise ValueError("Non-positive prices detected.")

if df.duplicated(["Date", "Ticker"]).any():
    raise ValueError("Duplicate ticker-date records detected.")

# One price column per stock
prices = df.pivot(
    index="Date",
    columns="Ticker",
    values="Close"
).sort_index()

tickers = ["AAPL", "MSFT", "NVDA"]
missing = [ticker for ticker in tickers if ticker not in prices.columns]

if missing:
    raise ValueError(f"Missing stock data: {missing}")

# Compare stocks on common dates
prices = prices[tickers].dropna(how="any")

if len(prices) < 3:
    raise ValueError("Not enough common price observations.")



In [ ]:
# 3. Calculate daily returns

returns = prices.pct_change(fill_method=None).iloc[1:]



In [6]:
# 4. Calculate volatility and related statistics

risk_rows = []

for ticker in tickers:
    daily = returns[ticker].dropna()

    daily_volatility = daily.std(ddof=1)
    annualized_volatility = (
        daily_volatility * np.sqrt(TRADING_DAYS_PER_YEAR)
    )

    annualized_mean_return = (
        daily.mean() * TRADING_DAYS_PER_YEAR
    )

    risk_rows.append({
        "Ticker": ticker,
        "Observations": len(daily),
        "Mean_Daily_Return_Pct": daily.mean() * 100,
        "Daily_Volatility_Pct": daily_volatility * 100,
        "Annualized_Volatility_Pct": annualized_volatility * 100,
        "Annualized_Mean_Return_Pct": annualized_mean_return * 100,
    })

risk_summary = pd.DataFrame(risk_rows)



In [7]:
# 5. Calculate drawdowns
# Normalize each stock to its own initial price.
# Running peaks identify the previous highest price.

running_peaks = prices.cummax()

drawdowns = prices.div(running_peaks).sub(1)

max_drawdown_rows = []

for ticker in tickers:
    series = drawdowns[ticker]

    trough_date = series.idxmin()
    max_drawdown = series.loc[trough_date]

    # Find the most recent peak at or before the trough.
    peak_prices = prices.loc[:trough_date, ticker]
    peak_date = peak_prices.idxmax()

    max_drawdown_rows.append({
        "Ticker": ticker,
        "Peak_Date": peak_date.date(),
        "Trough_Date": trough_date.date(),
        "Peak_Price": prices.loc[peak_date, ticker],
        "Trough_Price": prices.loc[trough_date, ticker],
        "Maximum_Drawdown_Pct": max_drawdown * 100,
    })

drawdown_summary = pd.DataFrame(max_drawdown_rows)

# Combine risk measures into one report
risk_summary = risk_summary.merge(
    drawdown_summary,
    on="Ticker",
    how="left"
)

risk_summary.to_csv(
    REPORT_DIR / "risk_summary.csv",
    index=False
)

returns.to_csv(REPORT_DIR / "daily_returns_for_risk.csv")
drawdowns.to_csv(REPORT_DIR / "drawdown_history.csv")



In [8]:
# 6. Chart: annualized volatility comparison

volatility_plot = risk_summary.set_index("Ticker")[
    "Annualized_Volatility_Pct"
].sort_values()

ax = volatility_plot.plot(
    kind="bar",
    figsize=(9, 5),
    rot=0
)

ax.set_title("Annualized Historical Volatility by Stock")
ax.set_xlabel("Stock")
ax.set_ylabel("Annualized Volatility (%)")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "annualized_volatility.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


In [9]:
# 7. Chart: drawdown history

fig, ax = plt.subplots(figsize=(12, 6))

for ticker in tickers:
    ax.plot(
        drawdowns.index,
        drawdowns[ticker] * 100,
        label=ticker,
        linewidth=1.4
    )

ax.set_title("Historical Drawdowns")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown from Previous Peak (%)")
ax.legend(title="Stock")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "drawdown_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


In [10]:
# 8. Print results

print("=" * 70)
print("STOCK MARKET RISK ANALYSIS")
print("=" * 70)

print(f"Analysis period: {prices.index[0].date()} "
      f"to {prices.index[-1].date()}")
print(f"Common price observations: {len(prices):,}")
print(f"Daily return observations: {len(returns):,}")

print("\nRISK SUMMARY")
print(risk_summary.round(2).to_string(index=False))


STOCK MARKET RISK ANALYSIS
Analysis period: 2021-10-22 to 2026-10-21
Common price observations: 1,254
Daily return observations: 1,253

RISK SUMMARY
Ticker  Observations  Mean_Daily_Return_Pct  Daily_Volatility_Pct  Annualized_Volatility_Pct  Annualized_Mean_Return_Pct  Peak_Date Trough_Date  Peak_Price  Trough_Price  Maximum_Drawdown_Pct
  AAPL          1253                   0.08                  1.77                      28.03                       21.29 2024-12-26  2025-04-08      257.15        171.37                -33.36
  MSFT          1253                   0.06                  1.78                      28.20                       15.73 2021-11-19  2022-11-03      329.89        207.34                -37.15
  NVDA          1253                   0.24                  3.27                      51.95                       59.30 2021-11-29  2022-10-14       33.23         11.19                -66.34
